## Criterio formal de selección del modelo final (IE12)

De acuerdo con la metodología del proyecto, el modelo definitivo se selecciona **exclusivamente a partir del desempeño en validación cruzada (5 folds, `random_state=42`)**, reservando el conjunto de prueba únicamente para reportar el rendimiento final:

1. **Criterio principal:** Menor **MAE promedio en validación cruzada** (en USD).
2. **Criterios de desempate (si la diferencia de MAE es menor a 200 USD o no es estadísticamente significativa):**
   - Menor desviación estándar del MAE entre folds (mayor estabilidad).
   - Menor RMSE en CV (mejor control de errores extremos).
   - Mayor cobertura dentro de tolerancia ($\pm 10\%$).
   - Simplicidad e interpretabilidad operacional.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import ttest_rel, wilcoxon

# Asegurar raíz del proyecto en sys.path
raiz = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

from src.train import (
    cargar_datos_crudos,
    cargar_mejores_parametros,
    construir_ridge_robusto,
    construir_stacking,
    crear_cv,
    metricas_por_fold,
    guardar_modelo,
    guardar_resultados,
)
from src.utils import MODELS

X_train, X_test, y_train, y_test = cargar_datos_crudos()
params = cargar_mejores_parametros()
cv = crear_cv()

# Modelos candidatos principales
modelo_ridge = construir_ridge_robusto(
    X_train,
    alpha=params["ridge_robusto"]["alpha"],
    recorte_z=params["ridge_robusto"]["recorte_z"],
)
modelo_stacking = construir_stacking(X_train, parametros=params)

# Evaluación fold a fold
folds_ridge = metricas_por_fold(modelo_ridge, X_train, y_train, cv=cv)
folds_stacking = metricas_por_fold(modelo_stacking, X_train, y_train, cv=cv)

comparacion_folds = pd.DataFrame({
    "fold": folds_stacking["fold"],
    "MAE_Ridge_robusto": folds_ridge["MAE"],
    "MAE_Stacking": folds_stacking["MAE"],
    "Diferencia (Stacking - Ridge)": folds_stacking["MAE"] - folds_ridge["MAE"],
})
display(comparacion_folds)

# Pruebas de significancia pareadas sobre el MAE por fold
stat_t, p_t = ttest_rel(folds_stacking["MAE"], folds_ridge["MAE"])
stat_w, p_w = wilcoxon(folds_stacking["MAE"], folds_ridge["MAE"])

print(f"MAE CV Ridge robusto : {folds_ridge['MAE'].mean():,.2f} ± {folds_ridge['MAE'].std(ddof=1):,.2f} USD")
print(f"MAE CV Stacking      : {folds_stacking['MAE'].mean():,.2f} ± {folds_stacking['MAE'].std(ddof=1):,.2f} USD")
print(f"Mejora media Stacking: {(folds_ridge['MAE'].mean() - folds_stacking['MAE'].mean()):,.2f} USD por fold")
print(f"Test t pareado       : t = {stat_t:.4f}, p-valor = {p_t:.4f}")
print(f"Test Wilcoxon        : W = {stat_w:.4f}, p-valor = {p_w:.4f}")

,fold,MAE_Ridge_robusto,MAE_Stacking,Diferencia (Stacking - Ridge)
0,1,12957.345893,12271.018514,-686.327379
1,2,13594.693464,13430.401865,-164.291600
2,3,14815.118763,14383.626659,-431.492104
3,4,13440.055837,12604.543545,-835.512292
4,5,15612.820257,15603.197447,-9.622809


MAE CV Ridge robusto : 14,084.01 ± 1,094.90 USD
MAE CV Stacking      : 13,658.56 ± 1,359.74 USD
Mejora media Stacking: 425.45 USD por fold
Test t pareado       : t = -2.7538, p-valor = 0.0512
Test Wilcoxon        : W = 0.0000, p-valor = 0.0625


In [2]:
# 1. Ajustar Stacking con todo el conjunto de entrenamiento y guardar best_model.joblib
stacking_final = construir_stacking(X_train, parametros=params)
stacking_final.fit(X_train, y_train)
ruta_modelo = guardar_modelo(stacking_final)
print(f"Modelo final guardado en: {ruta_modelo}")

# 2. Actualizar columna 'seleccionado' en models/results_supervised.csv
ruta_csv = MODELS / "results_supervised.csv"
df_res = pd.read_csv(ruta_csv)

# Verificar qué modelos existen en la tabla
print("Modelos en results_supervised.csv:", df_res["modelo"].tolist())

df_res["seleccionado"] = False
mask_stacking = df_res["modelo"].str.contains("Stacking", case=False, na=False)
assert mask_stacking.sum() == 1, f"Se esperaba 1 fila de Stacking, se encontraron {mask_stacking.sum()}"
df_res.loc[mask_stacking, "seleccionado"] = True

guardar_resultados(df_res, ruta_csv)
display(df_res[["modelo", "MAE_CV", "MAE_prueba", "Cobertura10_prueba", "seleccionado"]])

Modelo final guardado en: C:\Users\Admin\OneDrive\Escritorio\machine\parte 3\Proyecto_ML_housing\models\best_model.joblib
Modelos en results_supervised.csv: ['Referencia (mediana)', 'Ridge base', 'Ridge ajustado', 'Boosting base', 'Boosting ajustado', 'Ridge robusto', 'Boosting optimizado', 'Random Forest', 'Extra Trees', 'Voting', 'Stacking']


,modelo,MAE_CV,MAE_prueba,Cobertura10_prueba,seleccionado
0,Referencia (mediana),54122.839590,63838.100683,0.186007,False
1,Ridge base,16064.398992,14940.413403,0.750853,False
2,Ridge ajustado,16064.398992,14940.413403,0.750853,False
3,Boosting base,14916.655581,15623.869249,0.742321,False
4,Boosting ajustado,14634.924306,15349.591878,0.747440,False
5,Ridge robusto,14084.006843,14085.894162,0.771331,False
6,Boosting optimizado,14707.904208,13386.729345,0.778157,False
7,Random Forest,16127.637211,15705.892491,0.725256,False
8,Extra Trees,16079.348131,15763.747480,0.720137,False
9,Voting,13995.222848,13362.749575,0.776451,False


In [4]:
from pathlib import Path
import pandas as pd

ruta_seg = Path("models/results_segments.csv")
if not ruta_seg.exists() and Path("../models/results_segments.csv").exists():
  ruta_seg = Path("../models/results_segments.csv")

df_seg = pd.read_csv(ruta_seg)
print("Filas y columnas:", df_seg.shape)
print(df_seg.to_string())

Filas y columnas: (6, 11)
                                                  segmento  n_prueba       MAE_USD  error_relativo_mediano  cobertura_10  cobertura_IC95_inf  cobertura_IC95_sup  MAE_referencia_USD  mejora_MAE_vs_referencia cumple_cobertura_70_puntual IC95_incluye_70
0                              S1 Dúplex y multifamiliares        26  10232.867704                0.072136      0.615385            0.425345            0.775716        33632.500000                  0.695745                       False            True
1                                  S2 Antiguas y compactas       135   9046.118227                0.060085      0.733333            0.653036            0.800718        45949.548148                  0.803129                        True            True
2  S3 Un piso de tamaño medio, construcción de los años 60       151   8152.777677                0.043023      0.847682            0.781804            0.896308        24096.894040                  0.661667               